# eRayz YOLO Training — Warzone/BO7 Player Detection
---
Trainiert ein YOLOv8s Modell auf dem Roboflow Warzone Dataset (5.895 Bilder).

**ANLEITUNG:**
1. Klicke oben auf `Laufzeit > Laufzeittyp aendern > GPU (T4)`
2. Fuehre ALLE Zellen der Reihe nach aus (Shift+Enter)
3. Am Ende wird `warzone_v1.onnx` heruntergeladen
4. Lege die Datei in deinen `Downloads\backend\` Ordner
5. Der Aimbot findet sie automatisch

In [ ]:
# SCHRITT 1: Ultralytics + Roboflow installieren
!pip install -q ultralytics roboflow
print('OK: Ultralytics + Roboflow installiert')

In [ ]:
# SCHRITT 2: GPU Check
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNUNG: Keine GPU! Gehe zu Laufzeit > Laufzeittyp aendern > GPU')

In [ ]:
# SCHRITT 3: Roboflow Warzone Dataset herunterladen
# Dieses Dataset hat 5.895 Bilder mit Enemy-Labels aus Warzone
from roboflow import Roboflow

# Kostenloser API Key von Roboflow (erstelle Account auf roboflow.com)
# Gehe zu: roboflow.com > Settings > API Key > Kopieren
RF_API_KEY = 'DEIN_ROBOFLOW_API_KEY_HIER'  # <-- HIER EINFUEGEN!

rf = Roboflow(api_key=RF_API_KEY)
project = rf.workspace('ia-black-ops-7-gepb0').project('warzone-detection-pfrlp')
dataset = project.version(4).download('yolov8')
print(f'Dataset heruntergeladen: {dataset.location}')

In [ ]:
# SCHRITT 4: Training starten
# YOLOv8s = gute Balance zwischen Geschwindigkeit und Genauigkeit
# 50 Epochen = ca. 20-30 Minuten auf T4 GPU
from ultralytics import YOLO

model = YOLO('yolov8s.pt')  # Vortrainiertes Modell als Basis

results = model.train(
    data=f'{dataset.location}/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    patience=10,
    name='warzone_v1',
    # Augmentation fuer bessere Generalisierung
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,
    flipud=0.0,      # Kein vertikales Flip (Spieler stehen aufrecht)
    fliplr=0.5,       # Horizontales Flip OK
    mosaic=1.0,
    mixup=0.1,
)
print('Training fertig!')

In [ ]:
# SCHRITT 5: Ergebnisse anzeigen
from IPython.display import Image, display
import os

# Finde den Trainings-Ordner
train_dir = 'runs/detect/warzone_v1'
if not os.path.exists(train_dir):
    # Fallback
    for d in os.listdir('runs/detect'):
        if 'warzone' in d:
            train_dir = f'runs/detect/{d}'
            break

print(f'Ergebnisse in: {train_dir}')

# Zeige Metriken
if os.path.exists(f'{train_dir}/results.png'):
    display(Image(filename=f'{train_dir}/results.png', width=800))

# Zeige Confusion Matrix
if os.path.exists(f'{train_dir}/confusion_matrix.png'):
    display(Image(filename=f'{train_dir}/confusion_matrix.png', width=500))

# Zeige Predictions
if os.path.exists(f'{train_dir}/val_batch0_pred.jpg'):
    display(Image(filename=f'{train_dir}/val_batch0_pred.jpg', width=800))

In [ ]:
# SCHRITT 6: Als ONNX exportieren
best_model = YOLO(f'{train_dir}/weights/best.pt')

best_model.export(
    format='onnx',
    imgsz=640,
    simplify=True,
    dynamic=False,
)

# Kopiere mit lesbarem Namen
import shutil
onnx_path = f'{train_dir}/weights/best.onnx'
final_name = 'warzone_v1.onnx'
shutil.copy(onnx_path, final_name)

size_mb = os.path.getsize(final_name) / (1024*1024)
print(f'\nFERTIG: {final_name} ({size_mb:.1f} MB)')
print('Jetzt runterladen und in Downloads/backend/ legen!')

In [ ]:
# SCHRITT 7: Download
from google.colab import files
files.download('warzone_v1.onnx')
print('Download gestartet!')